# 49W — Speaker Diarization
Takes pre-transcribed JSON files and assigns speaker labels (SPEAKER_00, SPEAKER_01...) to each segment.

**Input:** `MyDrive/49w-bot/data/transcribed/*.json` (from local transcription)

**Output:** `MyDrive/49w-bot/data/diarized/*.json` (transcription + speaker turns)

**Runtime:** GPU (T4 or better) — Runtime → Change runtime type → T4 GPU

In [ ]:
!pip install -q pyannote.audio==3.4.0 "huggingface_hub==0.24.0"

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# ── CONFIG ─────────────────────────────────────────────────
AUDIO_DIR       = "/content/drive/MyDrive/49w-bot/data/audio"
TRANSCRIBED_DIR = "/content/drive/MyDrive/49w-bot/data/transcribed"
OUTPUT_DIR      = "/content/drive/MyDrive/49w-bot/data/diarized"
HF_TOKEN        = "hf_..."   # paste your token
NUM_SPEAKERS    = 3
# ────────────────────────────────────────────────────────────

In [ ]:
import torch
from huggingface_hub import login
from pyannote.audio import Pipeline
from pathlib import Path

Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

login(token=HF_TOKEN, add_to_git_credential=False)
print("Loading diarization pipeline...")
pipeline = Pipeline.from_pretrained("pyannote/speaker-diarization-3.1")
pipeline = pipeline.to(device)
print("Pipeline ready.")

In [ ]:
import json
from tqdm.notebook import tqdm

def assign_speakers(segments, diarization):
    """Assign speaker label to each segment by maximum overlap."""
    turns = [(turn.start, turn.end, speaker)
             for turn, _, speaker in diarization.itertracks(yield_label=True)]
    for seg in segments:
        best_speaker, best_overlap = "UNKNOWN", 0
        for t_start, t_end, speaker in turns:
            overlap = min(seg["end"], t_end) - max(seg["start"], t_start)
            if overlap > best_overlap:
                best_overlap, best_speaker = overlap, speaker
        seg["speaker"] = best_speaker
    return segments

def merge_turns(segments):
    """Merge consecutive segments from the same speaker."""
    merged = []
    for seg in segments:
        if merged and merged[-1]["speaker"] == seg["speaker"]:
            merged[-1]["text"] += " " + seg["text"]
            merged[-1]["end"] = seg["end"]
        else:
            merged.append(dict(seg))
    return merged

transcribed_files = sorted(Path(TRANSCRIBED_DIR).glob("*.json"))
print(f"Found {len(transcribed_files)} transcribed files")

done, skipped, failed = 0, 0, 0

for t_path in tqdm(transcribed_files, desc="Diarizing"):
    vid_id = t_path.stem
    out_file = Path(OUTPUT_DIR) / f"{vid_id}.json"
    audio_path = Path(AUDIO_DIR) / f"{vid_id}.wav"

    if out_file.exists():
        skipped += 1
        continue
    if not audio_path.exists():
        print(f"  No audio: {vid_id}")
        failed += 1
        continue

    try:
        with open(t_path, encoding="utf-8") as f:
            record = json.load(f)

        diarization = pipeline(str(audio_path), num_speakers=NUM_SPEAKERS)
        segments = assign_speakers(record["segments"], diarization)
        turns = merge_turns(segments)

        out = {
            "video_id": vid_id,
            "title": record["title"],
            "turns": turns,
            "speakers": list(set(t["speaker"] for t in turns)),
        }
        with open(out_file, "w", encoding="utf-8") as f:
            json.dump(out, f, ensure_ascii=False, indent=2)
        done += 1

    except Exception as e:
        print(f"  Failed {vid_id}: {e}")
        failed += 1

print(f"\nDone: {done} | Skipped: {skipped} | Failed: {failed}")

In [ ]:
# Preview first diarized video
sample = next(Path(OUTPUT_DIR).glob("*.json"))
with open(sample) as f:
    rec = json.load(f)
print(f"Title: {rec['title']}")
print(f"Speakers found: {rec['speakers']}")
print(f"\nFirst 10 turns:")
for t in rec["turns"][:10]:
    print(f"  [{t['speaker']}] {t['text'][:80]}")